In [2]:
!pip install -q langgraph langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.2 MB/s eta 0:00:00


In [4]:
import os
from typing import TypedDict, List, Dict, Any, Optional
from pprint import pprint

#using OpenAI
from langchain_openai import ChatOpenAI
import os

#https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = "sk-proj-8dHOTK9NVnmCFg00QA8zbboTnOfC1FLuMsA"


llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)



In [5]:
# Build the Business Environment
#Step 0 — Mock Enterprise Data
INCIDENTS = {
    "INC-2045": {
        "incident_id": "INC-2045",
        "application": "Checkout-Service",
        "server": "APP-PROD-12",
        "severity": "P1",
        "description": "Checkout response time increased from 1.2 sec to 8.7 sec"
    }
}

# Monitoring system:
SERVER_METRICS = {
    "APP-PROD-12": {
        "cpu": 96,
        "memory": 71,
        "db_connections": 100,
        "error_rate": 18,
        "status": "DEGRADED"
    }
}

# Historical incidents:
HISTORICAL_INCIDENTS = [
    {
        "incident_id": "INC-1781",
        "symptom": "High latency and DB connections at 100%",
        "root_cause": "Database connection pool exhaustion",
        "resolution": "Restart application connection pool and increase pool size"
    },
    {
        "incident_id": "INC-1654",
        "symptom": "High CPU and slow checkout",
        "root_cause": "Expensive SQL query",
        "resolution": "Optimize SQL query and add missing database index"
    }
]

#Runbooks:
RUNBOOKS = {
    "connection_pool": """
    1. Check active DB connections.
    2. Verify connection pool utilization.
    3. Check database availability.
    4. Restart application connection pool if approved.
    5. Monitor latency after remediation.
    """,

    "high_cpu": """
    1. Identify high CPU process.
    2. Check recent deployments.
    3. Inspect database queries.
    4. Validate application thread usage.
    """
}



In [6]:
#Step 1 — Build Tools: Now create ordinary Python functions.
#Tool 1 — Incident lookup
def get_incident(incident_id):
    return INCIDENTS.get(
        incident_id,
        {"error": "Incident not found"}
    )

#Tool 2 — Server health
def get_server_metrics(server):
    return SERVER_METRICS.get(
        server,
        {"error": "Server not found"}
    )

#Tool 3 — Previous incidents
def search_previous_incidents(keyword):
    keyword = keyword.lower()

    results = []

    for incident in HISTORICAL_INCIDENTS:

        text = str(incident).lower()

        if keyword in text:
            results.append(incident)

    return results

#Tool 4 — Runbook
def search_runbook(topic):

    topic = topic.lower()

    if "connection" in topic:
        return RUNBOOKS["connection_pool"]

    if "cpu" in topic:
        return RUNBOOKS["high_cpu"]

    return "No matching runbook found."



In [7]:
# Check point 1: Before agents, execute the tools manually.
incident = get_incident("INC-2045")
pprint(incident)

metrics = get_server_metrics(
    incident["server"]
)

pprint(metrics)

{'application': 'Checkout-Service',
 'description': 'Checkout response time increased from 1.2 sec to 8.7 sec',
 'incident_id': 'INC-2045',
 'server': 'APP-PROD-12',
 'severity': 'P1'}
{'cpu': 96,
 'db_connections': 100,
 'error_rate': 18,
 'memory': 71,
 'status': 'DEGRADED'}


In [8]:
#Step 2 — Basic LLM
prompt = """
Incident:

Checkout response time increased
from 1.2 seconds to 8.7 seconds.

What is the likely root cause?
"""

response = llm.invoke(prompt)

print(response.content)

An increase in checkout response time from 1.2 seconds to 8.7 seconds is significant and suggests a performance bottleneck or failure in one or more components involved in the checkout process. The likely root causes could include:

1. **Database Performance Issues**  
   - Slow queries or locks on checkout-related tables (e.g., orders, inventory, payments).  
   - Increased load causing database contention or resource exhaustion.  
   - Indexes missing or corrupted, leading to slower data retrieval.

2. **Third-Party Service Latency**  
   - Payment gateway or fraud detection services responding slowly or timing out.  
   - External APIs (e.g., shipping, tax calculation) experiencing delays.

3. **Application Server Bottlenecks**  
   - Increased CPU or memory usage causing slower processing.  
   - Thread pool exhaustion or blocking calls in the checkout service.  
   - Recent code changes introducing inefficient logic or loops.

4. **Network Issues**  
   - Increased latency or pack

In [9]:
#Step 3 — Add State

#Create a state object.
class IncidentState(TypedDict):
    incident_id: str
    incident: Optional[Dict]
    metrics: Optional[Dict]
    previous_incidents: List[Dict]
    runbook: Optional[str]
    observations: List[str]
    hypotheses: List[str]
    recommendation: Optional[str]
    confidence: float
    approval_status: str


state = {
    "incident_id": "INC-2045",
    "incident": None,
    "metrics": None,
    "previous_incidents": [],
    "runbook": None,
    "observations": [],
    "hypotheses": [],
    "recommendation": None,
    "confidence": 0.0,
    "approval_status": "NOT_REQUIRED"
}

In [10]:
#Step 4 — Build Deterministic Workflow Nodes

#Node 1 — Load incident
def load_incident(state):

    incident = get_incident(
        state["incident_id"]
    )

    state["incident"] = incident

    state["observations"].append(
        "Incident loaded"
    )

    return state

#Node 2 — Collect metrics
def collect_metrics(state):

    server = state["incident"]["server"]

    metrics = get_server_metrics(server)

    state["metrics"] = metrics

    state["observations"].append(
        f"Server metrics collected: {metrics}"
    )

    return state

#Node 3 — Search memory
def retrieve_history(state):

    results = search_previous_incidents(
        "connection pool"
    )

    state["previous_incidents"] = results

    state["observations"].append(
        f"Found {len(results)} similar incidents"
    )

    return state


#Node 4 — Retrieve runbook
def retrieve_runbook(state):

    runbook = search_runbook(
        "connection pool"
    )

    state["runbook"] = runbook

    state["observations"].append(
        "Relevant runbook retrieved"
    )

    return state




In [12]:
#Step 5 — Diagnosis Node

def diagnose(state):

    prompt = f"""
  You are an enterprise IT incident diagnosis agent.

  Incident: {state["incident"]}

  Metrics: {state["metrics"]}

  Previous incidents: {state["previous_incidents"]}

  Runbook: {state["runbook"]}

  Determine:
  1. Most likely root cause
  2. Evidence
  3. Confidence from 0 to 1
  4. Recommended next action

Be concise.
"""

    response = llm.invoke(prompt)
    state["recommendation"] = response.content

    return state


In [14]:
#Step 6 — Build the LangGraph Workflow
from langgraph.graph import StateGraph, END

#create workflow
workflow = StateGraph(IncidentState)

#Add nodes:
workflow.add_node(
    "load_incident",
    load_incident
)

workflow.add_node(
    "collect_metrics",
    collect_metrics
)

workflow.add_node(
    "retrieve_history",
    retrieve_history
)

workflow.add_node(
    "retrieve_runbook",
    retrieve_runbook
)

workflow.add_node(
    "diagnose",
    diagnose
)


#Add workflow edges (one by one)
workflow.set_entry_point(
    "load_incident"
)

workflow.add_edge(
    "load_incident",
    "collect_metrics"
)

workflow.add_edge(
    "collect_metrics",
    "retrieve_history"
)

workflow.add_edge(
    "retrieve_history",
    "retrieve_runbook"
)

workflow.add_edge(
    "retrieve_runbook",
    "diagnose"
)

workflow.add_edge(
    "diagnose",
    END
)

#compile
app = workflow.compile()

#Run
result = app.invoke(state)

pprint(result)



{'approval_status': 'NOT_REQUIRED',
 'confidence': 0.0,
 'hypotheses': [],
 'incident': {'application': 'Checkout-Service',
              'description': 'Checkout response time increased from 1.2 sec to '
                             '8.7 sec',
              'incident_id': 'INC-2045',
              'server': 'APP-PROD-12',
              'severity': 'P1'},
 'incident_id': 'INC-2045',
 'metrics': {'cpu': 96,
             'db_connections': 100,
             'error_rate': 18,
             'memory': 71,
             'status': 'DEGRADED'},
 'observations': ['Incident loaded',
                  "Server metrics collected: {'cpu': 96, 'memory': 71, "
                  "'db_connections': 100, 'error_rate': 18, 'status': "
                  "'DEGRADED'}",
                  'Found 1 similar incidents',
                  'Relevant runbook retrieved'],
 'previous_incidents': [{'incident_id': 'INC-1781',
                         'resolution': 'Restart application connection pool '
                   

In [15]:
#Step 7 — Add Conditional Reasoning
#Now make the workflow respond to evidence.

def decide_next_step(state):

    metrics = state["metrics"]

    if metrics["db_connections"] >= 95:
        return "database_issue"

    if metrics["cpu"] >= 90:
        return "cpu_issue"

    return "unknown_issue"

#Database investigation
def investigate_database(state):

    state["hypotheses"].append(
        "Possible database connection pool exhaustion"
    )

    state["runbook"] = search_runbook(
        "connection pool"
    )

    return state


#CPU investigation:
def investigate_cpu(state):

    state["hypotheses"].append(
        "Possible CPU saturation"
    )

    state["runbook"] = search_runbook(
        "high cpu"
    )

    return state

#Unknown:
def gather_more_evidence(state):

    state["observations"].append(
        "No dominant signal. More investigation required."
    )

    return state

#Now add conditional edges.
workflow.add_conditional_edges(
    "collect_metrics",
    decide_next_step,
    {
        "database_issue": "investigate_database",
        "cpu_issue": "investigate_cpu",
        "unknown_issue": "gather_more_evidence"
    }
)

#This is the beginning of orchestration.


In [17]:
#Step 8 — Human Approval
def assess_risk(state):

    text = (
        state["recommendation"]
        or ""
    ).lower()

    if "restart" in text:
        state["approval_status"] = \
            "REQUIRED"

    else:
        state["approval_status"] = \
            "NOT_REQUIRED"

    return state


#Decision:
def approval_route(state):

    if state["approval_status"] == "REQUIRED":
        return "approval"

    return "complete"


#Add a mock human approval node.
def human_approval(state):

    print(
        "\nHuman approval required."
    )

    print(
        state["recommendation"]
    )

    decision = input(
        "Approve? yes/no: "
    )

    if decision.lower() == "yes":
        state["approval_status"] = "APPROVED"

    else:
        state["approval_status"] = "REJECTED"

    return state

In [18]:
#Step 9 — Specialized Agents
#Monitoring Agent
def monitoring_agent(state):

    incident = state["incident"]

    metrics = get_server_metrics(
        incident["server"]
    )

    return {
        "metrics": metrics
    }


#Knowledge Agent:
def knowledge_agent(state):

    history = search_previous_incidents(
        "connection pool"
    )

    runbook = search_runbook(
        "connection pool"
    )

    return {
        "history": history,
        "runbook": runbook
    }


#Diagnosis Agent:
def diagnosis_agent(
    incident,
    metrics,
    history,
    runbook
):

    prompt = f"""
Diagnose the incident.

Incident:
{incident}

Metrics:
{metrics}

History:
{history}

Runbook:
{runbook}

Return:
root cause,
evidence,
confidence,
recommendation.
"""

    return llm.invoke(prompt).content


#Supervisor:
def supervisor_agent(
    incident_id
):

    incident = get_incident(
        incident_id
    )

    monitoring = monitoring_agent(
        {"incident": incident}
    )

    knowledge = knowledge_agent(
        {"incident": incident}
    )

    diagnosis = diagnosis_agent(
        incident,
        monitoring["metrics"],
        knowledge["history"],
        knowledge["runbook"]
    )

    return {
        "incident": incident,
        "metrics":
            monitoring["metrics"],
        "knowledge":
            knowledge,
        "diagnosis":
            diagnosis
    }


#Run:
result = supervisor_agent(
    "INC-2045"
)

pprint(result)

{'diagnosis': 'root cause: Database connection pool exhaustion causing '
              'resource contention and increased response time.\n'
              '\n'
              'evidence:  \n'
              '- DB connections at 100% indicating full utilization of the '
              'connection pool.  \n'
              '- High CPU usage at 96% and elevated error rate at 18% suggest '
              'system strain.  \n'
              '- Incident description notes response time increased from 1.2 '
              'sec to 8.7 sec, consistent with resource bottleneck.  \n'
              '- Previous similar incident (INC-1781) had the same symptom and '
              'root cause.\n'
              '\n'
              'confidence: High\n'
              '\n'
              'recommendation:  \n'
              '1. Verify current connection pool utilization and confirm DB '
              'availability.  \n'
              '2. Restart the application connection pool to free up '
              'connections 

In [19]:
#Step 10 — Add Memory

#HISTORICAL_INCIDENTS

CURRENT INCIDENT <br>
       ↓ <br>
Create retrieval query <br>
       ↓ <br>
HISTORICAL MEMORY <br>
       ↓ <br>
Retrieve similar incidents <br>
       ↓ <br>
Add to current context <br>
       ↓ <br>
DIAGNOSIS <br>

In [21]:
#Step 11 — Observability
#Create an event log.
execution_log = []

#Function:
import time

def log_event(
    agent,
    action,
    tool=None,
    input_data=None,
    output_data=None,
    error=None
):

    execution_log.append({
        "timestamp":
            time.time(),

        "agent":
            agent,

        "action":
            action,

        "tool":
            tool,

        "input":
            input_data,

        "output":
            output_data,

        "error":
            error
    })


#Use:
log_event(
    agent="MonitoringAgent",
    action="Collect metrics",
    tool="get_server_metrics",
    input_data="APP-PROD-12",
    output_data=SERVER_METRICS[
        "APP-PROD-12"
    ]
)

pprint(execution_log)

[{'action': 'Collect metrics',
  'agent': 'MonitoringAgent',
  'error': None,
  'input': 'APP-PROD-12',
  'output': {'cpu': 96,
             'db_connections': 100,
             'error_rate': 18,
             'memory': 71,
             'status': 'DEGRADED'},
  'timestamp': 1788794764.8520758,
  'tool': 'get_server_metrics'}]


In [ ]:
#Step 12 — Add Simple Retry
#Create a deliberately unreliable tool.
import random

def unreliable_monitoring(server):

    if random.random() < 0.4:
        raise Exception(
            "Monitoring API unavailable"
        )

    return get_server_metrics(server)


#Retry wrapper:
def call_with_retry(
    function,
    *args,
    max_retries=3
):

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:
            return function(*args)

        except Exception as error:

            print(
                f"Attempt {attempt} failed:",
                error
            )

            if attempt == max_retries:
                raise

#Run
metrics = call_with_retry(
    unreliable_monitoring,
    "APP-PROD-12"
)


In [24]:
#RUN
# Create a fresh state for the final run
final_state = {
    "incident_id": "INC-2045",

    "incident": {
        "incident_id": "INC-2045",
        "application": "Checkout-Service",
        "server": "APP-PROD-12",
        "severity": "P1",
        "description": "Checkout response time increased from 1.2 sec to 8.7 sec"
    },

    "metrics": {
        "cpu": 96,
        "memory": 71,
        "db_connections": 100,
        "error_rate": 18,
        "status": "DEGRADED"
    },

    "previous_incidents": [
        {
            "incident_id": "INC-1781",
            "symptom": "High latency and DB connections at 100%",
            "root_cause": "Database connection pool exhaustion",
            "resolution": "Restart application connection pool and increase pool size"
        }
    ],

    "runbook": """
1. Check active DB connections.
2. Verify connection pool utilization.
3. Check database availability.
4. Restart application connection pool if approved.
5. Monitor latency after remediation.
""",

    "observations": [
        "Incident INC-2045 loaded",
        "Checkout latency increased from 1.2 sec to 8.7 sec",
        "CPU utilization is critically high at 96%",
        "Database connections have reached 100%",
        "Application error rate is elevated at 18%",
        "Server status is DEGRADED",
        "Similar historical incident INC-1781 was found",
        "Relevant connection-pool runbook retrieved"
    ],

    "hypotheses": [
        "Primary hypothesis: Database connection pool exhaustion",
        "Secondary contributing factor: High CPU utilization"
    ],

    "root_cause": "Database connection pool exhaustion",

    "evidence": [
        "Database connections are at 100%",
        "Application status is DEGRADED",
        "Error rate is 18%",
        "Historical incident INC-1781 showed the same symptoms",
        "Historical root cause was database connection pool exhaustion"
    ],

    "confidence": 0.90,

    "recommendation": (
        "Verify connection pool utilization and database availability. "
        "If confirmed, restart the application connection pool and review "
        "connection pool sizing. Monitor checkout latency, error rate and "
        "database connections after remediation."
    ),

    "risk_level": "HIGH",

    "approval_status": "REQUIRED",

    "next_action": "WAIT_FOR_HUMAN_APPROVAL",

    "status": "WAITING_FOR_APPROVAL"
}

# Run the complete agent workflow
result = app.invoke(final_state)

pprint(result)

{'approval_status': 'REQUIRED',
 'confidence': 0.9,
 'hypotheses': ['Primary hypothesis: Database connection pool exhaustion',
                'Secondary contributing factor: High CPU utilization'],
 'incident': {'application': 'Checkout-Service',
              'description': 'Checkout response time increased from 1.2 sec to '
                             '8.7 sec',
              'incident_id': 'INC-2045',
              'server': 'APP-PROD-12',
              'severity': 'P1'},
 'incident_id': 'INC-2045',
 'metrics': {'cpu': 96,
             'db_connections': 100,
             'error_rate': 18,
             'memory': 71,
             'status': 'DEGRADED'},
 'observations': ['Incident INC-2045 loaded',
                  'Checkout latency increased from 1.2 sec to 8.7 sec',
                  'CPU utilization is critically high at 96%',
                  'Database connections have reached 100%',
                  'Application error rate is elevated at 18%',
                  'Server status


**Final Capstone Run**

Input:

Investigate INC-2045

The final screen should show something like:

[Supervisor]
Investigation started.

[Incident Tool]
INC-2045 loaded.

[Monitoring Agent]
CPU: 96%
DB Connections: 100%
Status: DEGRADED

[Knowledge Agent]
Similar incident:
INC-1781

Root cause:
Connection pool exhaustion.

[Diagnosis Agent]
Most likely root cause:
Database connection pool exhaustion.

Confidence:
0.87

Recommended action:
Restart application connection pool
and review pool sizing.

[Policy]
Restart requires approval.

[Status]
WAITING_FOR_APPROVAL

Then:

Engineer approves.

Continue:

[Action]
Remediation authorized.

[Verification]
Latency decreased.

[Status]
RESOLVED